In [1]:
# SPDX-FileCopyrightText: Copyright (c) 2025-2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
# SPDX-License-Identifier: Apache-2.0

# ---
# jupyter:
#   jupytext:
#     text_representation:
#       extension: .py
#       format_name: percent
#       format_version: '1.3'
#   kernelspec:
#     display_name: Python 3
#     language: python
#     name: python3
# ---

# 🕵️ Choosing a Replacement Strategy

Four [replace mode](../../concepts/replace/) strategies compared side-by-side on the same data.

| Strategy | What it does |
|----------|-------------|
| **Substitute** | LLM-generated contextual replacements |
| **Redact** | Label-based markers (`[REDACTED_FIRST_NAME]`) |
| **Annotate** | Tags entities but keeps original text |
| **Hash** | Deterministic hash digest |

#### 📚 What you'll learn

- Compare **Redact**, **Annotate**, **Hash**, and **Substitute** on the same input
- Customize output formats with `format_template`
- Understand which strategy fits your use case (readability, determinism, privacy)

> **Tip:** First time running notebooks? Start with
> [setup instructions](https://nvidia-nemo.github.io/Anonymizer/latest/tutorials/).

## ⚙️ Setup

- Check if your `NVIDIA_API_KEY` from [build.nvidia.com](https://build.nvidia.com) is registered for model access.
  - The default `build.nvidia.com` (NVIDIA Build) setup is a convenient way to try Anonymizer and iterate on previews. Use of NVIDIA Build is subject to NVIDIA Build's own terms of service and privacy practices, which are separate from and independent of the NeMo Framework library. NVIDIA Build is intended for evaluation and testing purposes only and may not be used in production environments. Do not upload any confidential information or personal data when using NVIDIA Build. Your use of NVIDIA Build is logged for security purposes and to improve NVIDIA products and services.
  - Request and token rate limits on `build.nvidia.com` vary by account and model access, and lower-volume development access can be slow for full-dataset runs. Start with `preview()` on a small sample, then move to your own endpoint for production data and usage.
- Import all four strategy classes: `Redact`, `Annotate`, `Hash`, `Substitute`.
- `Anonymizer()` initializes with the default model provider -- no extra config needed.
- `configure_logging(LoggingConfig.default())` keeps logs at INFO. Switch to `LoggingConfig.debug()` when troubleshooting.

In [2]:
import getpass
import os

if not os.getenv("NVIDIA_API_KEY"):
    key = getpass.getpass("Enter NVIDIA_API_KEY from build.nvidia.com: ").strip()
    if not key:
        raise RuntimeError("NVIDIA_API_KEY is required to run these notebooks.")
    os.environ["NVIDIA_API_KEY"] = key

In [3]:
from anonymizer import (
    Annotate,
    Anonymizer,
    AnonymizerConfig,
    AnonymizerInput,
    Hash,
    LoggingConfig,
    Redact,
    Substitute,
    configure_logging,
)

configure_logging(LoggingConfig.default())

In [4]:
anonymizer = Anonymizer()

[11:07:15] [INFO] 🔧 Anonymizer initialized with 3 model configs


[11:07:15] [INFO]   |-- 🔎 detector:  gliner-pii-detector


[11:07:15] [INFO]   |-- ✅ validator: gpt-oss-120b


[11:07:15] [INFO]   |-- 🧩 augmenter: gpt-oss-120b


## 📦 Input data

- We use the same biographies dataset throughout so each strategy is compared
  on identical input.

In [5]:
input_data = AnonymizerInput(
    source="https://raw.githubusercontent.com/NVIDIA-NeMo/Anonymizer/refs/heads/main/docs/data/NVIDIA_synthetic_biographies.csv",
    text_column="biography",
    data_summary="Biographical profiles",
)

## 🔄 Substitute

- Uses an LLM to generate contextually appropriate synthetic replacements.
  - The LLM considers the full document context matching names with emails, cities to states, etc.
- Customize with `instructions` to steer the LLM's replacement choices.

In [6]:
substitute_config = AnonymizerConfig(replace=Substitute())

substitute_preview = anonymizer.preview(
    config=substitute_config,
    data=input_data,
    num_records=3,
)

[11:07:16] [INFO] 👀 Preview mode: 📂 Loaded 3 records from https://raw.githubusercontent.com/NVIDIA-NeMo/Anonymizer/refs/heads/main/docs/data/NVIDIA_synthetic_biographies.csv (column: 'biography')


[11:07:16] [INFO] 🔍 Running entity detection on 3 records


[11:07:16] [INFO] detection labels in scope: (default: 65 labels; see anonymizer.DEFAULT_ENTITY_LABELS for list)


[11:08:17] [INFO]   |-- 📋 Detection complete — 79 entities found across 3 records (0 failed) [61.4s]


[11:08:17] [INFO]   |-- labels: first_name=22, organization_name=7, age=5, occupation=5, city=4, state=4, degree=4, university=4, field_of_study=4, last_name=3, race_ethnicity=3, political_view=3, language=2, religious_belief=2, street_address=2, place_name=1, date_of_birth=1, proprietary_term=1, employment_status=1, company_name=1


[11:08:17] [INFO] 🔄 Running Substitute replacement


[11:08:58] [INFO]   |-- 📋 Replacement complete (0 failed) [40.9s]


[11:08:58] [INFO] 🎉 Pipeline complete — 3 records processed, 0 total failures


In [7]:
substitute_preview.display_record(0)

Original,Label,Replacement
40,age,45
Aria and Leo,first_name,Nina and Omar
Bobby,first_name,Ethan
Christian Democrat,political_view,Libertarian
Colorado,state,Oregon
Colorado Veterinary Clinic,organization_name,Oregon Veterinary Center
DVM,degree,Doctor of Osteopathic Medicine (DO)
Denver,city,Portland
English,language,Spanish
Jefferson High,organization_name,Willamette High


### Custom instructions

- Pass `instructions` to guide the LLM -- e.g. keep replacements within
  a specific region, culture, or naming convention.

In [8]:
substitute_custom_config = AnonymizerConfig(
    replace=Substitute(instructions="Use only Japanese names and locations for all replacements.")
)
substitute_custom_preview = anonymizer.preview(
    config=substitute_custom_config,
    data=input_data,
    num_records=3,
)
substitute_custom_preview.display_record(0)

[11:08:59] [INFO] 👀 Preview mode: 📂 Loaded 3 records from https://raw.githubusercontent.com/NVIDIA-NeMo/Anonymizer/refs/heads/main/docs/data/NVIDIA_synthetic_biographies.csv (column: 'biography')


[11:08:59] [INFO] 🔍 Running entity detection on 3 records


[11:08:59] [INFO] detection labels in scope: (default: 65 labels; see anonymizer.DEFAULT_ENTITY_LABELS for list)


[11:09:44] [INFO]   |-- 📋 Detection complete — 77 entities found across 3 records (0 failed) [45.2s]


[11:09:44] [INFO]   |-- labels: first_name=22, organization_name=6, age=5, occupation=5, city=4, state=4, degree=4, university=4, field_of_study=4, last_name=3, race_ethnicity=3, political_view=3, language=2, religious_belief=2, street_address=2, education_level=1, place_name=1, date_of_birth=1, employment_status=1


[11:09:44] [INFO] 🔄 Running Substitute replacement


[11:10:03] [INFO]   |-- 📋 Replacement complete (0 failed) [18.9s]


[11:10:03] [INFO] 🎉 Pipeline complete — 3 records processed, 0 total failures


Original,Label,Replacement
40,age,45
Aria and Leo,first_name,Kenji and Yui
Bobby,first_name,Haruto
Christian Democrat,political_view,Liberal Democratic Party member
Colorado,state,Hokkaido
Colorado Veterinary Clinic,organization_name,Hokkaido Veterinary Center
DVM,degree,獣医学修士
Denver,city,Sapporo
English,language,Japanese
Jefferson High,education_level,Sapporo High School


## 🚫 Redact

- Replaces each entity with a label-based marker. Default: `[REDACTED_FIRST_NAME]`.
- Customize with `Redact(format_template=...)`.

In [9]:
redact_config = AnonymizerConfig(replace=Redact())

redact_preview = anonymizer.preview(
    config=redact_config,
    data=input_data,
    num_records=3,
)

redact_preview.display_record(0)

[11:10:04] [INFO] 👀 Preview mode: 📂 Loaded 3 records from https://raw.githubusercontent.com/NVIDIA-NeMo/Anonymizer/refs/heads/main/docs/data/NVIDIA_synthetic_biographies.csv (column: 'biography')


[11:10:04] [INFO] 🔍 Running entity detection on 3 records


[11:10:04] [INFO] detection labels in scope: (default: 65 labels; see anonymizer.DEFAULT_ENTITY_LABELS for list)


[11:10:33] [INFO]   |-- 📋 Detection complete — 76 entities found across 3 records (0 failed) [29.8s]


[11:10:33] [INFO]   |-- labels: first_name=22, organization_name=7, age=5, occupation=5, city=4, state=4, degree=4, university=4, field_of_study=4, last_name=3, race_ethnicity=3, political_view=3, language=2, street_address=2, place_name=1, date_of_birth=1, employment_status=1, religious_belief=1


[11:10:33] [INFO] 🔄 Running Redact replacement


[11:10:33] [INFO]   |-- 📋 Replacement complete (0 failed) [0.0s]


[11:10:33] [INFO] 🎉 Pipeline complete — 3 records processed, 0 total failures


Original,Label,Replacement
Bobby,first_name,[REDACTED_FIRST_NAME]
Watford,last_name,[REDACTED_LAST_NAME]
40,age,[REDACTED_AGE]
Mexican,race_ethnicity,[REDACTED_RACE_ETHNICITY]
veterinarian,occupation,[REDACTED_OCCUPATION]
Denver,city,[REDACTED_CITY]
Colorado,state,[REDACTED_STATE]
Jefferson High,organization_name,[REDACTED_ORGANIZATION_NAME]
DVM,degree,[REDACTED_DEGREE]
University of Colorado Boulder,university,[REDACTED_UNIVERSITY]


### Custom template

- `format_template="***"` replaces every entity with the same constant.

In [10]:
custom_config = AnonymizerConfig(replace=Redact(format_template="***"))

custom_preview = anonymizer.preview(
    config=custom_config,
    data=input_data,
    num_records=3,
)

custom_preview.display_record(0)

[11:10:34] [INFO] 👀 Preview mode: 📂 Loaded 3 records from https://raw.githubusercontent.com/NVIDIA-NeMo/Anonymizer/refs/heads/main/docs/data/NVIDIA_synthetic_biographies.csv (column: 'biography')


[11:10:34] [INFO] 🔍 Running entity detection on 3 records


[11:10:34] [INFO] detection labels in scope: (default: 65 labels; see anonymizer.DEFAULT_ENTITY_LABELS for list)


[11:11:06] [INFO]   |-- 📋 Detection complete — 77 entities found across 3 records (0 failed) [32.2s]


[11:11:06] [INFO]   |-- labels: first_name=22, organization_name=7, age=5, occupation=5, city=4, state=4, degree=4, university=4, field_of_study=4, last_name=3, race_ethnicity=3, political_view=3, language=2, religious_belief=2, street_address=2, place_name=1, date_of_birth=1, employment_status=1


[11:11:06] [INFO] 🔄 Running Redact replacement


[11:11:06] [INFO]   |-- 📋 Replacement complete (0 failed) [0.0s]


[11:11:06] [INFO] 🎉 Pipeline complete — 3 records processed, 0 total failures


Original,Label,Replacement
Bobby,first_name,***
Watford,last_name,***
40,age,***
Mexican,race_ethnicity,***
veterinarian,occupation,***
Denver,city,***
Colorado,state,***
Jefferson High,organization_name,***
DVM,degree,***
University of Colorado Boulder,university,***


## 🏷️ Annotate

- Tags each entity with its label but keeps the original text visible.
  Default: `<Alice, first_name>`.
- Customize with `format_template` -- must include `{text}` and `{label}`,
  e.g. `Annotate(format_template="<{text}-|-{label}>")`.

In [11]:
annotate_config = AnonymizerConfig(replace=Annotate())

annotate_preview = anonymizer.preview(
    config=annotate_config,
    data=input_data,
    num_records=3,
)

annotate_preview.display_record(0)

[11:11:07] [INFO] 👀 Preview mode: 📂 Loaded 3 records from https://raw.githubusercontent.com/NVIDIA-NeMo/Anonymizer/refs/heads/main/docs/data/NVIDIA_synthetic_biographies.csv (column: 'biography')


[11:11:07] [INFO] 🔍 Running entity detection on 3 records


[11:11:07] [INFO] detection labels in scope: (default: 65 labels; see anonymizer.DEFAULT_ENTITY_LABELS for list)


[11:11:37] [INFO]   |-- 📋 Detection complete — 78 entities found across 3 records (0 failed) [29.8s]


[11:11:37] [INFO]   |-- labels: first_name=22, organization_name=7, age=5, occupation=5, city=4, state=4, degree=4, university=4, field_of_study=4, last_name=3, race_ethnicity=3, political_view=3, language=2, religious_belief=2, street_address=2, place_name=1, date_of_birth=1, employment_status=1, company_name=1


[11:11:37] [INFO] 🔄 Running Annotate replacement


[11:11:37] [INFO]   |-- 📋 Replacement complete (0 failed) [0.0s]


[11:11:37] [INFO] 🎉 Pipeline complete — 3 records processed, 0 total failures


Original,Label,Replacement
Bobby,first_name,"<Bobby, first_name>"
Watford,last_name,"<Watford, last_name>"
40,age,"<40, age>"
Mexican,race_ethnicity,"<Mexican, race_ethnicity>"
veterinarian,occupation,"<veterinarian, occupation>"
Denver,city,"<Denver, city>"
Colorado,state,"<Colorado, state>"
Jefferson High,organization_name,"<Jefferson High, organization_name>"
DVM,degree,"<DVM, degree>"
University of Colorado Boulder,university,"<University of Colorado Boulder, university>"


### Custom template

- Override the default format with any string containing `{text}` and `{label}`.

In [12]:
annotate_custom_config = AnonymizerConfig(replace=Annotate(format_template="<{text}-|-{label}>"))
annotate_custom_preview = anonymizer.preview(
    config=annotate_custom_config,
    data=input_data,
    num_records=3,
)
annotate_custom_preview.display_record(0)

[11:11:37] [INFO] 👀 Preview mode: 📂 Loaded 3 records from https://raw.githubusercontent.com/NVIDIA-NeMo/Anonymizer/refs/heads/main/docs/data/NVIDIA_synthetic_biographies.csv (column: 'biography')


[11:11:37] [INFO] 🔍 Running entity detection on 3 records


[11:11:37] [INFO] detection labels in scope: (default: 65 labels; see anonymizer.DEFAULT_ENTITY_LABELS for list)


[11:12:12] [INFO]   |-- 📋 Detection complete — 76 entities found across 3 records (0 failed) [35.3s]


[11:12:12] [INFO]   |-- labels: first_name=23, organization_name=7, age=5, occupation=5, city=4, state=4, degree=4, university=4, field_of_study=4, last_name=3, race_ethnicity=3, language=2, political_view=2, religious_belief=2, street_address=2, place_name=1, date_of_birth=1


[11:12:12] [INFO] 🔄 Running Annotate replacement


[11:12:12] [INFO]   |-- 📋 Replacement complete (0 failed) [0.0s]


[11:12:12] [INFO] 🎉 Pipeline complete — 3 records processed, 0 total failures


Original,Label,Replacement
Bobby,first_name,<Bobby-|-first_name>
Watford,last_name,<Watford-|-last_name>
40,age,<40-|-age>
Mexican,race_ethnicity,<Mexican-|-race_ethnicity>
veterinarian,occupation,<veterinarian-|-occupation>
Denver,city,<Denver-|-city>
Colorado,state,<Colorado-|-state>
Jefferson High,organization_name,<Jefferson High-|-organization_name>
DVM,degree,<DVM-|-degree>
University of Colorado Boulder,university,<University of Colorado Boulder-|-university>


## #️⃣ Hash

- Deterministic -- same input always produces the same hash.
- Customize with `format_template` (must include `{digest}`),
  `algorithm` (`sha256`/`sha1`/`md5`), and `digest_length` (6-64 characters).

In [13]:
hash_config = AnonymizerConfig(replace=Hash())

hash_preview = anonymizer.preview(
    config=hash_config,
    data=input_data,
    num_records=3,
)

hash_preview.display_record(0)

[11:12:13] [INFO] 👀 Preview mode: 📂 Loaded 3 records from https://raw.githubusercontent.com/NVIDIA-NeMo/Anonymizer/refs/heads/main/docs/data/NVIDIA_synthetic_biographies.csv (column: 'biography')


[11:12:13] [INFO] 🔍 Running entity detection on 3 records


[11:12:13] [INFO] detection labels in scope: (default: 65 labels; see anonymizer.DEFAULT_ENTITY_LABELS for list)


[11:12:44] [INFO]   |-- 📋 Detection complete — 78 entities found across 3 records (0 failed) [30.9s]


[11:12:44] [INFO]   |-- labels: first_name=22, organization_name=8, age=5, occupation=5, city=4, state=4, degree=4, university=4, field_of_study=4, political_view=4, last_name=3, race_ethnicity=3, language=2, street_address=2, place_name=1, date_of_birth=1, employment_status=1, religious_belief=1


[11:12:44] [INFO] 🔄 Running Hash replacement


[11:12:44] [INFO]   |-- 📋 Replacement complete (0 failed) [0.0s]


[11:12:44] [INFO] 🎉 Pipeline complete — 3 records processed, 0 total failures


Original,Label,Replacement
Bobby,first_name,<HASH_FIRST_NAME_4a70dab2cb4d>
Watford,last_name,<HASH_LAST_NAME_e2efa8a62600>
40,age,<HASH_AGE_d59eced1ded0>
Mexican,race_ethnicity,<HASH_RACE_ETHNICITY_d108dfd1df5c>
veterinarian,occupation,<HASH_OCCUPATION_52a469e4d8e9>
Denver,city,<HASH_CITY_fcdeb8c07d4a>
Colorado,state,<HASH_STATE_4ae62bf4e804>
Jefferson High,organization_name,<HASH_ORGANIZATION_NAME_39dde416149c>
DVM,degree,<HASH_DEGREE_d44ae5e206d1>
University of Colorado Boulder,university,<HASH_UNIVERSITY_bca201129c41>


### Custom template

- Override the algorithm, digest length, and output format.

In [14]:
hash_custom_config = AnonymizerConfig(replace=Hash(algorithm="md5", digest_length=8, format_template="[{digest}]"))
hash_custom_preview = anonymizer.preview(
    config=hash_custom_config,
    data=input_data,
    num_records=3,
)
hash_custom_preview.display_record(0)

[11:12:45] [INFO] 👀 Preview mode: 📂 Loaded 3 records from https://raw.githubusercontent.com/NVIDIA-NeMo/Anonymizer/refs/heads/main/docs/data/NVIDIA_synthetic_biographies.csv (column: 'biography')


[11:12:45] [INFO] 🔍 Running entity detection on 3 records


[11:12:45] [INFO] detection labels in scope: (default: 65 labels; see anonymizer.DEFAULT_ENTITY_LABELS for list)


[11:13:20] [INFO]   |-- 📋 Detection complete — 78 entities found across 3 records (0 failed) [35.4s]


[11:13:20] [INFO]   |-- labels: first_name=22, organization_name=8, age=5, occupation=5, city=4, state=4, degree=4, university=4, field_of_study=4, last_name=3, race_ethnicity=3, political_view=3, language=2, religious_belief=2, street_address=2, place_name=1, date_of_birth=1, employment_status=1


[11:13:20] [INFO] 🔄 Running Hash replacement


[11:13:20] [INFO]   |-- 📋 Replacement complete (0 failed) [0.0s]


[11:13:20] [INFO] 🎉 Pipeline complete — 3 records processed, 0 total failures


Original,Label,Replacement
Bobby,first_name,[657b3da9]
Watford,last_name,[6e424e2c]
40,age,[d645920e]
Mexican,race_ethnicity,[a0e769d8]
veterinarian,occupation,[84c99b4a]
Denver,city,[67100af8]
Colorado,state,[15e49475]
Jefferson High,organization_name,[27c56955]
DVM,degree,[47211f54]
University of Colorado Boulder,university,[e2b97348]


## 📊 (Optional) Evaluate each strategy

- `evaluate()` is a separate, opt-in step that scores the output with LLM-as-judge metrics. Which metrics fire depends on the strategy:
  - **Substitute** → 4 metrics (Detection Validity + Type Fidelity + Relational Consistency + Attribute Fidelity).
  - **Redact / Annotate / Hash** → Detection Validity only (no replacement map to score type/relational/attribute against).
- Below shows it on the Substitute preview to surface all four; the same call works on `redact_preview`, `annotate_preview`, or `hash_preview`.

In [15]:
substitute_evaluated = anonymizer.evaluate(substitute_preview)
substitute_evaluated.display_record(0)

## ⏭️ Next steps

- **[🕵️ Inspecting Detected Entities](../02_inspecting_detected_entities/)** --
  dig into what the detection pipeline found and debug quality.
- **[✏️ Rewriting Biographies](../04_rewriting_biographies/)** --
  generate privacy-safe paraphrases instead of token-level replacements.
- **[⚖️ Rewriting Legal Documents](../05_rewriting_legal_documents/)** --
  rewrite legal text with domain-specific privacy goals.